# Stage One — Data Inventory and Reliability Assessment

## Project overview

Telecommunications providers lose customers to competitors on a continuous basis. The industry term for this is **churn**, and it represents a significant commercial cost, as acquiring a new customer is considerably more expensive than retaining an existing one.

This project predicts which customers are likely to depart, and then addresses a second question which most churn projects omit: **whether a given customer is worth the cost of retaining them.** For a substantial proportion of the customer base, the answer is no.

The work is delivered in four stages: data assessment, database construction and customer value, model development, and intervention economics. Each stage is documented in a notebook of its own, of which this is the first. The four notebooks form one continuous account, read in sequence.

## Purpose of this stage

The dataset is supplied by IBM and describes the customer base of a telecommunications provider in California. It arrives in a finished state, with two figures already calculated for every customer: a churn score representing IBM's own prediction of departure, and a customer lifetime value figure.

Both figures appear authoritative and both save considerable effort. That is precisely the risk. Where a supplied figure is unsound, all work built upon it is also unsound, and the defect remains concealed because the figure is not questioned.

Three assessments are therefore performed before any development work begins:

1. **A structural review** — the number of records, the fields present, and how each field is stored
2. **An assessment of missing values** — where data is absent, and whether the absence can be accounted for
3. **A reliability test of the supplied figures** — whether they measure what they purport to measure

**Scope.** No data is altered at this stage. This notebook examines and records only. All corrections are applied at Stage Two, implemented in SQL, where they remain visible and open to inspection.

---

## 1. Retrieving the dataset

The dataset is retrieved programmatically rather than downloaded manually. This ensures that any party running the project obtains identical data from an identical source, with no dependency upon a file held locally. Work which cannot be reproduced cannot be verified.

The complete version of the dataset is used in preference to the abridged version which circulates more widely. The abridged version omits the customer lifetime value figure, the recorded reason for departure, and the geographic detail, all of which this project requires.

In [1]:
import kagglehub, os

path = kagglehub.dataset_download("yeanzc/telco-customer-churn-ibm-dataset")
print("Path:", path)
print(os.listdir(path))

c:\Users\IC Clearwater\OneDrive\Documents\GitHub\Customer_Lifetime_Value_Churn_Prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path: C:\Users\IC Clearwater\.cache\kagglehub\datasets\yeanzc\telco-customer-churn-ibm-dataset\versions\1
['Telco_customer_churn.xlsx']


## 2. Confirming the structure of the dataset

Load the file, and confirm the number of records together with the fields present.

The field names are printed before any further work is undertaken. This project depends upon the presence of specific fields; were any absent, that must be established immediately rather than discovered after substantial work has been completed. Verification takes seconds, whereas an unfounded assumption can cost considerably more.

The first ten records are also displayed, in order to establish the format of the values and identify any obvious irregularities.

In [2]:
import pandas as pd

df = pd.read_excel(f"{path}/Telco_customer_churn.xlsx")
print(df.shape)
print(df.columns.tolist())
pd.set_option('display.max_columns', None)
df.head(10)

(7043, 33)
['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices
5,4190-MFLUW,1,United States,California,Los Angeles,90020,"34.066367, -118.309868",34.066367,-118.309868,Female,No,Yes,No,10,Yes,No,DSL,No,No,Yes,Yes,No,No,Month-to-month,No,Credit card (automatic),55.20,528.35,Yes,1,78,5925,Competitor offered higher download speeds
6,8779-QRDMV,1,United States,California,Los Angeles,90022,"34.02381, -118.156582",34.023810,-118.156582,Male,Yes,No,No,1,No,No phone service,DSL,No,No,Yes,No,No,Yes,Month-to-month,Yes,Electronic check,39.65,39.65,Yes,1,100,5433,Competitor offered more data
7,1066-JKSGK,1,United States,California,Los Angeles,90024,"34.066303, -118.435479",34.066303,-118.435479,Male,No,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.15,20.15,Yes,1,92,4832,Competitor made better offer
8,6467-CHFZW,1,United States,California,Los Angeles,90028,"34.099869, -118.326843",34.099869,-118.326843,Male,No,Yes,Yes,47,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.35,4749.15,Yes,1,77,5789,Competitor had better devices
9,8665-UTDHZ,1,United States,California,Los Angeles,90029,"34.089953, -118.294824",34.089953,-118.294824,Male,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,No,Electronic check,30.20,30.2,Yes,1,97,2915,Competitor had better devices


**Result:** 7,043 customers across 33 fields, consistent with expectations.

The 33 fields cover five categories of information: customer characteristics, location, products held, financial arrangements, and the departure outcome.

## 3. Reviewing field types and missing values

Two checks are performed. The first establishes how each field is stored — as a number, a date, or as text. The second counts missing values, restricted to those fields where any occur.

The field types matter for a specific reason. A field containing monetary amounts must be stored as a number so that it can be aggregated and averaged. Where such a field is stored as text, this indicates that at least one entry cannot be interpreted as a number, and requires investigation.

In [3]:
pd.set_option('display.max_rows', 40)
print(df.dtypes)
print("\nNulls:")
print(df.isna().sum()[df.isna().sum() > 0])

CustomerID               str
Count                  int64
Country                  str
State                    str
City                     str
Zip Code               int64
Lat Long                 str
Latitude             float64
Longitude            float64
Gender                   str
Senior Citizen           str
Partner                  str
Dependents               str
Tenure Months          int64
Phone Service            str
Multiple Lines           str
Internet Service         str
Online Security          str
Online Backup            str
Device Protection        str
Tech Support             str
Streaming TV             str
Streaming Movies         str
Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges         object
Churn Label              str
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason             str
dtype: object

Nulls:
Churn Reason    5174


**Result:** missing values are confined to a single field, the recorded reason for departure, which is absent for 5,174 records.

On initial inspection this represents a substantial gap affecting nearly three quarters of the dataset. Examination of the figures establishes otherwise. The dataset contains 7,043 customers, of whom 1,869 have departed. This leaves 5,174 customers still with the business — precisely the number of absent values.

The field is therefore populated only where a customer has actually departed. Where a customer remains, no reason exists to be recorded. **The field is not incomplete; it is correctly empty, and no treatment is applied.**

The principle demonstrated is that the cause of a gap should be established before any decision is taken regarding whether to fill it. Populating these values automatically, as is standard in much data preparation, would have fabricated a reason for departure for 5,174 customers who have not departed.

## 4. Completing the field type inventory

The previous output was truncated on display. The final ten fields are printed separately to ensure that no field goes unexamined.

A truncated output means fields were not inspected, and an uninspected field constitutes an assumption.

In [4]:
print(df.dtypes.tail(10))

Contract                 str
Paperless Billing        str
Payment Method           str
Monthly Charges      float64
Total Charges         object
Churn Label              str
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason             str
dtype: object


**Result:** monthly charges is stored as a number, whereas total charges is stored as text.

Both fields contain monetary amounts and both should therefore be numeric. The cause is specific: where a single entry within a column cannot be interpreted as a number, the entire column is treated as text. One unreadable value among 7,043 records is sufficient to render the whole field unusable for calculation.

## 5. Investigating the non-numeric entries

The column is not converted directly. Converting first and investigating afterwards would destroy the evidence required to establish the underlying cause.

Each value is instead tested individually, with those which fail conversion flagged and isolated. The affected records are then examined against tenure and departure status, in order to establish what those customers have in common.

In [5]:
bad = pd.to_numeric(df['Total Charges'], errors='coerce').isna()
print(bad.sum())
print(df.loc[bad, ['Tenure Months', 'Monthly Charges', 'Total Charges', 'Churn Label']])

11
      Tenure Months  Monthly Charges Total Charges Churn Label
2234              0            52.55                        No
2438              0            20.25                        No
2568              0            80.85                        No
2667              0            25.75                        No
2856              0            56.05                        No
4331              0            19.85                        No
4687              0            25.35                        No
5104              0            20.00                        No
5719              0            19.70                        No
6772              0            73.35                        No
6840              0            61.90                        No


**Result:** eleven records are affected, and all eleven share an identical profile — zero months of tenure, a monthly charge on record, and active customer status.

The explanation follows directly. These are newly acquired customers who have agreed a price but have not yet been billed for a full cycle. No total charge exists because no charge has yet been raised. **The blank value is therefore accurate rather than corrupt.**

**Decision: substitute zero, and retain all eleven records.**

Zero is the correct figure, as these customers have genuinely paid nothing to date. Deleting the records would have been the faster course, and is what commonly occurs in practice. It would also have removed the entire population of newly acquired customers from the analysis — and as Stage Two establishes, that population is the one most likely to depart. Their removal would have introduced a systematic bias before the substantive work had commenced.

## 6. Correcting the field and summarising the numeric data

Convert total charges to a numeric field, substituting zero for the eleven unbilled accounts, then produce summary statistics for the principal numeric fields.

Summary statistics comprise the count, average, spread, minimum, maximum and quarter-point values of each field. The purpose is to establish the range and distribution of every numeric field at once, which allows implausible values to be identified.

In [6]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce').fillna(0)
print(df['Total Charges'].dtype)
print(df[['Tenure Months','Monthly Charges','Total Charges','CLTV']].describe())

float64
       Tenure Months  Monthly Charges  Total Charges         CLTV
count    7043.000000      7043.000000    7043.000000  7043.000000
mean       32.371149        64.761692    2279.734304  4400.295755
std        24.559481        30.090047    2266.794470  1183.057152
min         0.000000        18.250000       0.000000  2003.000000
25%         9.000000        35.500000     398.550000  3469.000000
50%        29.000000        70.350000    1394.550000  4527.000000
75%        55.000000        89.850000    3786.600000  5380.500000
max        72.000000       118.750000    8684.800000  6500.000000


**Result:** total charges is now numeric and available for calculation.

The summary raises a concern regarding the supplied customer lifetime value figure. **It has a minimum of 2,003 and a maximum of 6,500, while total charges — representing revenue already collected — reaches 8,684.**

This combination is not possible. Customer lifetime value represents the total worth of a customer across the entire relationship. It cannot be lower than revenue the business has already received from that customer.

The boundaries provide a second indication. No customer scores below 2,003 or above 6,500. Genuine customer value does not exhibit this pattern; any real customer base contains customers of negligible value and customers of exceptional value.

## 7. Testing the reliability of the supplied lifetime value figure

Two independent indications of unreliability warrant a formal test rather than a judgement.

Irrespective of the formula applied, customer lifetime value is determined by two variables: the amount a customer pays per period, and the duration of the relationship. Any legitimate figure must therefore vary in accordance with both.

This provides a testable proposition: **does the supplied figure rank customers in the same order as its own constituent variables?**

**Rank correlation** is applied rather than a measure of scale. Rank correlation establishes only whether two variables order a set of items in the same sequence, disregarding scale entirely. Disregarding scale is the intention here, as the figure is already suspected of being expressed on an arbitrary scale. The result is expressed between −1 and 1, where a result approaching 1 indicates near-identical ranking and a result approaching 0 indicates no relationship.

In [7]:
print(df[['CLTV','Monthly Charges','Total Charges','Tenure Months','Churn Value']].corr(method='spearman')['CLTV'])

CLTV               1.000000
Monthly Charges    0.107944
Total Charges      0.310721
Tenure Months      0.367039
Churn Value       -0.123627
Name: CLTV, dtype: float64


**Result:** all four correlations fall materially below the threshold required of a valid measure.

| Supplied figure tested against | Result | Expected for a valid measure |
|---|---|---|
| Total charges | **0.31** | Strong positive |
| Tenure | **0.37** | Strong positive |
| Monthly charge | **0.11** | Strong positive |
| Departure status | **−0.12** | Clearly negative |

**The third result is determinative.** The monthly charge is a direct and primary constituent of customer lifetime value; a doubling of the monthly charge should produce an approximate doubling of value. The recorded relationship is 0.11, indicating effectively no relationship at all.

Considered alongside the impossible boundaries identified above, the conclusion is not in doubt.

**Decision: the supplied lifetime value figure is rejected.** It is a score expressed on an arbitrary scale rather than a currency measure. Customer value is derived independently at Stage Two, calculated from actual monthly charges, actual tenure, and a stated cost-to-serve assumption, with every assumption documented.

**The derived figure is deliberately not validated against the supplied figure.** Once a measure has been established as unreliable, it cannot serve as a benchmark for the work replacing it.

---

## Summary of findings

**1. Dataset dimensions.** 7,043 customer records across 33 fields, of which 1,869 customers have departed.

**2. Missing values.** Confined to the recorded reason for departure and fully accounted for by departure status. No treatment required.

**3. Data quality.** Total charges held blank values for 11 unbilled new customers, forcing the field into text format. Corrected to numeric with zero substituted, and all records retained.

**4. Supplied lifetime value rejected.** The figure carries impossible boundaries, falls below revenue already collected for certain customers, and bears negligible relationship to its own constituent variables. Customer value is derived independently at Stage Two.

**5. Leakage identified.** The churn score and the recorded reason for departure are both excluded from the model. The churn score is a prediction produced by another model and supplied with the dataset; the reason for departure exists only after the event it would be used to predict. Either would reveal the outcome and produce a misleadingly favourable result. The reason for departure is retained within the database for reporting purposes.

**6. Fields carrying no information.** Three fields hold an identical value across every record, and one duplicates data already present in separate fields. All four are excluded from the schema.

**7. A recording issue requiring explicit treatment.** Six service fields hold three values rather than two, the third denoting that the service was never available rather than declined. The distinction is preserved and addressed explicitly at Stage Two.

**Next stage:** construct the relational schema, derive the required measures, and calculate customer value in SQL.